# Data Merging Sandbox

This is where I will work to merge datasets for different sites (Gothic and Kettle Ponds) and external gridded datasets at hourly to daily resolutions.

In [27]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
project_root = "/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo"
if project_root not in sys.path:
    sys.path.append(project_root)
from utils import process_sail_data

DATA_PATH = '/storage/dlhogan/precipitation-rodeo/data/processed/'

# Sand Castle 1 - Gothic Precipitation

In [60]:
# Load datasets from Gothic
billy_barr_ds = xr.open_dataset(f'{DATA_PATH}billy_barr/billy_barr_20211001-20230930_30min.nc')[['precip','precip_bad_flag','precip_missing_flag',]].sortby('time')
# rename flags to qc_missing_bb and qc_bad_bb
billy_barr_ds = billy_barr_ds.rename({'precip_missing_flag':'qc_missing_billy_barr_precip',
                                      'precip_bad_flag':'qc_bad_billy_barr_precip'})
sail_ld_ds = xr.open_dataset(f'{DATA_PATH}SAIL/laser_disdrometer_gothic_processed_30min.nc')[['precip_accum_unadjusted','precip_accum_holyroyd',
                                                                                             'precip_accum_brandes','precip_accum_heymsfield',
                                                                                             'precip_missing_flag','precip_bad_flag',]].sortby('time')
# rename variables to qc_missing_sail_ld and qc_bad_sail_ld
sail_ld_ds = sail_ld_ds.rename({'precip_missing_flag':'qc_missing_sail_ld_unadjusted',
                                'precip_bad_flag':'qc_bad_sail_ld_unadjusted'})
sail_pluvio_ds = xr.open_dataset(f'{DATA_PATH}SAIL/pluvio_30min.nc')[['accum_nrt','pluvio_missing_flag','pluvio_bad_flag',]].sortby('time')
# rename variables to qc_missing_sail_pluvio and qc_bad_sail_pluvio
sail_pluvio_ds = sail_pluvio_ds.rename({'pluvio_missing_flag':'qc_missing_sail_pluvio',
                                        'pluvio_bad_flag':'qc_bad_sail_pluvio'})
# load sail met and squire datasets
sail_met_ds = xr.open_dataset(f'{DATA_PATH}SAIL/met_30min.nc').sortby('time')
sail_squire_ds = xr.open_dataset(f'{DATA_PATH}SAIL/squire_30min.nc').sel(site='gothic').sortby('time').squeeze()

# drop all vars in sail_met_ds not in process_sail_data.SAIL_PRECIPITATION_VARS['cumulative'] 
met_prcp_vars = [var for var in process_sail_data.SAIL_PRECIPITATION_VARS['cumulative'] if var in sail_met_ds.data_vars]
# add qc variables for sail_met_ds
met_prcp_vars += [f'{var}_missing_flag' for var in ['org_precip', 'pwd_precip', 'tbrg_precip']]
met_prcp_vars += [f'{var}_bad_flag' for var in ['org_precip', 'pwd_precip', 'tbrg_precip']]
sail_met_ds = sail_met_ds[met_prcp_vars]
# rename sail met qc variables
sail_met_ds = sail_met_ds.rename({'org_precip_missing_flag':'qc_missing_sail_org',
                                    'org_precip_bad_flag':'qc_bad_sail_org',
                                    'pwd_precip_missing_flag':'qc_missing_sail_pwd',
                                    'pwd_precip_bad_flag':'qc_bad_sail_pwd',
                                    'tbrg_precip_missing_flag':'qc_missing_sail_tbg',
                                    'tbrg_precip_bad_flag':'qc_bad_sail_tbg',})

squire_prcp_vars = [var for var in process_sail_data.SAIL_PRECIPITATION_VARS['cumulative'] if var in sail_squire_ds.data_vars]
squire_prcp_vars += ["squire_missing_flag","squire_bad_flag"]
sail_squire_ds = sail_squire_ds[squire_prcp_vars].drop_vars(['lat','lon','x','y','site'])
# rename sail squire qc variables
sail_squire_ds = sail_squire_ds.rename({'squire_missing_flag':'qc_missing_sail_squire_m2009_1',
                                        'squire_bad_flag':'qc_bad_sail_squire_m2009_1'})


In [61]:
print('Merging datasets...')
try:
    gothic_combined_ds = xr.merge([billy_barr_ds, sail_ld_ds, sail_pluvio_ds, sail_met_ds, sail_squire_ds], compat='override')
    print('Datasets merged successfully.')
except Exception as e:
    print(f'Error merging datasets: {e}')

Merging datasets...
Datasets merged successfully.


In [ ]:
# rename the variables
variable_renames = {
    'precip': 'billy_barr_precip',
    'precip_accum_unadjusted': 'sail_ld_unadjusted',
    'precip_accum_holyroyd':'sail_ld_holyroyd',
    'precip_accum_brandes':'sail_ld_brandes',
    'precip_accum_heymsfield':'sail_ld_heymsfield',
    'accum_nrt':'sail_pluvio',
    'pwd_precip_total':'sail_pwd',
    'tbrg_precip_total':'sail_tbg',
    'org_precip_accum':'sail_org',
    'snow_rate_m2009_1_total':'sail_squire_m2009_1',
    'snow_rate_m2009_2_total':'sail_squire_m2009_2',
    'snow_rate_ws88diw_total':'sail_squire_ws88diw',
    'snow_rate_ws2012_total':'sail_squire_ws2012'
}

gothic_combined_ds = gothic_combined_ds.rename(variable_renames)

# update all bad flags to 1 where data is nan
for var in gothic_combined_ds.data_vars:
    if 'qc_bad' in var:
        data_var = var.replace('qc_bad_', '')
        gothic_combined_ds[var] = gothic_combined_ds[var].where(~gothic_combined_ds[data_var].isnull(), 1)


/home/dlhogan/miniforge3/envs/data-and-plotting/lib/python3.13/site-packages/xarray/core/duck_array_ops.py:251: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


# Sand Castle 2 - Kettle Ponds Precipitation

In [67]:
# Load datasets from Kettle p\Ponds
splash_lpdf_ds = xr.open_dataset(f'{DATA_PATH}SPLASH/lpdf_gauge_30min.nc')
# rename qc_variables
splash_lpds_ds = splash_lpdf_ds.rename({'qc_missing_precip':'qc_missing_splash_pluvio',
                                        'qc_bad_precip':'qc_bad_splash_pluvio'})
splash_ld_ds = xr.open_dataset(f'{DATA_PATH}SPLASH/SPLASH_kp_laser_disdrometer_30min.nc')
sos_ds = xr.open_dataset(f'{DATA_PATH}SOS/sos_ds_30min.nc')[['SWE_p1_c_max_accum', 'qc_SWE_p1_c_missing', 'qc_SWE_p1_c_bad',
                                                             'SWE_p2_c_max_accum', 'qc_SWE_p2_c_missing', 'qc_SWE_p2_c_bad',
                                                             'SWE_p3_c_max_accum', 'qc_SWE_p3_c_missing', 'qc_SWE_p3_c_bad',
                                                             'SWE_p4_c_max_accum', 'qc_SWE_p4_c_missing', 'qc_SWE_p4_c_bad']].sortby('time')
# rename qc variable to remove _c
sos_ds = sos_ds.rename({var: var.replace('_c_', '_') for var in sos_ds.data_vars if 'qc_' in var})


In [72]:
splash_ld_ds.data_vars

Data variables:
    Amount                    (time) float64 271kB ...
    Rate                      (time) float64 271kB ...
    precip_type               (time) float64 271kB ...
    qc_missing_precip         (time) int64 271kB ...
    qc_bad_precip             (time) int64 271kB ...
    precip_accum_unadjusted  (time) float64 271kB ...
    precip_accum_brandes      (time) float64 271kB ...
    precip_accum_heymsfield   (time) float64 271kB ...
    precip_accum_holyroyd     (time) float64 271kB ...
    precip_rate_holyroyd      (time) float64 271kB ...
    precip_rate_unadjusted   (time) float64 271kB ...
    precip_rate_brandes       (time) float64 271kB ...
    precip_rate_heymsfield    (time) float64 271kB ...

In [66]:
# merge kettle ponds datasets
print('Merging Kettle Ponds datasets...')
try:
    kettle_ponds_combined_ds = xr.merge([splash_lpdf_ds, splash_ld_ds, sos_ds], compat='override')
    print('Kettle Ponds datasets merged successfully.')
except Exception as e:
    print(f'Error merging Kettle Ponds datasets: {e}')

Merging Kettle Ponds datasets...
Kettle Ponds datasets merged successfully.


In [ ]:
variable_renames_kp = {
    'prcp':'splash_lpdf',
    'Amount': 'splash_ld_unadjusted',
    'SWE_p1_c_max_accum': "sos_swe_p1",
    'SWE_p2_c_max_accum': "sos_swe_p2",
    'SWE_p3_c_max_accum': "sos_swe_p3",
    'SWE_p4_c_max_accum': "sos_swe_p4"
}

kettle_ponds_combined_ds = kettle_ponds_combined_ds.rename(variable_renames_kp)